# Codex final: thesis-decision relaxation ladder

This is the final Colab/Kaggle runner for the Garrido-Rios MFSC decision ladder. It follows the scientific conclusion from the audit:

- The variables and value sets are thesis-faithful: `I_t` with `t in {168, 336, 504, 672, 1344}` and `S in {1, 2, 3}`.
- The experimental protocol is not thesis-identical once we cross `I x S`, allow per-node `t`, or re-decide weekly.
- Each relaxation is run as a separate ladder step, and adaptive PPO is compared against the best static policy in the same action space.

Default action today: `RUN_PROFILE = "l1a_today"`, which runs the 18 common-period `I x S` configurations with 30 replications.


In [ ]:
# =====================
# 0) Run config
# =====================

RUN_PROFILE = "l1a_today"  # smoke, l1a_today, l0_l1a_full, overnight_l1b, ppo_e2a_debug, ppo_e2a_serious, ppo_e2b_serious

GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "main"
REPO_DIR_COLAB = "/content/scresia"
REPO_DIR_KAGGLE = "/kaggle/working/scresia"

USE_GOOGLE_DRIVE_OUTPUT = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/scresia_results/codex_final_thesis_ladder"
LOCAL_OUTPUT_DIR = "/content/codex_final_thesis_ladder"
KAGGLE_OUTPUT_DIR = "/kaggle/working/codex_final_thesis_ladder"

RUNTIME_PIP_PACKAGES = [
    "simpy>=4.1",
    "numpy>=1.26",
    "gymnasium>=0.29",
    "stable-baselines3>=2.3",
    "sb3-contrib>=2.3",
    "pandas>=2.2",
]

BASE_SEED = 42
REWARD_MODE = "ReT_cd_v1"
RISK_LEVEL = "increased"
OBSERVATION_VERSION = "v5"
OBSERVATION_MODE = "env_sdm_history_reward"
STEP_SIZE_HOURS = 168.0
STOCHASTIC_PT = True
GARRIDO_CFIS = "31-90"

PROFILE_CONFIG = {
    "smoke": dict(kind="static", levels=["L0_garrido", "L1a_uniform_IxS"], replications=1, garrido_cfis="31", max_steps=4),
    "l1a_today": dict(kind="static", levels=["L1a_uniform_IxS"], replications=30, garrido_cfis=GARRIDO_CFIS, max_steps=260),
    "l0_l1a_full": dict(kind="static", levels=["L0_garrido", "L1a_uniform_IxS"], replications=30, garrido_cfis=GARRIDO_CFIS, max_steps=260),
    "overnight_l1b": dict(kind="static", levels=["L1b_per_node_IxS"], replications=30, garrido_cfis=GARRIDO_CFIS, max_steps=260, l1b_screening_replications=5, l1b_top_k=20),
    "ppo_e2a_debug": dict(kind="ppo", action_space_mode="thesis_factorized", inventory_period_mode="thesis_strict", timesteps=512, eval_episodes=1, seeds=[101], max_steps=8),
    "ppo_e2a_serious": dict(kind="ppo", action_space_mode="thesis_factorized", inventory_period_mode="thesis_strict", timesteps=200_000, eval_episodes=5, seeds=[101, 202, 303], max_steps=260),
    "ppo_e2b_serious": dict(kind="ppo", action_space_mode="factorized", inventory_period_mode="per_node", timesteps=200_000, eval_episodes=5, seeds=[101, 202, 303], max_steps=260),
}

cfg = PROFILE_CONFIG[RUN_PROFILE]
print({"RUN_PROFILE": RUN_PROFILE, **cfg})


In [ ]:
# ==========================================
# 1) Portable setup: Colab, Kaggle, or local
# ==========================================

from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/working").exists()
if not (IN_COLAB or IN_KAGGLE):
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {cmd}")
    return result

if IN_COLAB and USE_GOOGLE_DRIVE_OUTPUT:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True, timeout_ms=120_000)
        drive_mounted = True
    except Exception as exc:
        print("WARNING: Drive mount failed; using /content outputs.", repr(exc))
        drive_mounted = False
else:
    drive_mounted = False

if IN_COLAB or IN_KAGGLE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", *RUNTIME_PIP_PACKAGES])

if IN_COLAB:
    repo_root = Path(REPO_DIR_COLAB)
    shutil.rmtree(repo_root, ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(repo_root)])
    output_root = Path(DRIVE_OUTPUT_DIR if drive_mounted else LOCAL_OUTPUT_DIR)
elif IN_KAGGLE:
    repo_root = Path(REPO_DIR_KAGGLE)
    shutil.rmtree(repo_root, ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(repo_root)])
    output_root = Path(KAGGLE_OUTPUT_DIR)
else:
    cwd = Path.cwd().resolve()
    repo_root = cwd if (cwd / "supply_chain").exists() else cwd.parent
    output_root = repo_root / "outputs" / "codex_final_thesis_ladder"

sys.path.insert(0, str(repo_root.parent))
sys.path.insert(0, str(repo_root))
output_root.mkdir(parents=True, exist_ok=True)

git_hash = run_cmd(["git", "rev-parse", "HEAD"], cwd=repo_root).stdout.strip()
print("repo_root:", repo_root)
print("output_root:", output_root)
print("git_hash:", git_hash)
print("python:", sys.version)
print("platform:", platform.platform())


In [ ]:
# ======================================
# 2) Verify required repo contracts
# ======================================

import pandas as pd

ppo_help = run_cmd([sys.executable, "scripts/run_thesis_decision_ppo_smoke.py", "--help"], cwd=repo_root).stdout
ladder_help = run_cmd([sys.executable, "scripts/run_thesis_decision_ladder.py", "--help"], cwd=repo_root).stdout
assert "thesis_factorized" in ppo_help, "PPO script lacks thesis_factorized action mode."
assert "--inventory-period-mode" in ppo_help, "PPO script lacks explicit inventory_period_mode."
assert "L1a_uniform_IxS" in ladder_help and "L1b_per_node_IxS" in ladder_help, "Ladder script lacks required static levels."
print("contract check: ok")


In [ ]:
# ==========================================
# 3) Run selected static ladder or PPO profile
# ==========================================

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dirs = []

if cfg["kind"] == "static":
    run_label = f"{RUN_PROFILE}_{run_stamp}"
    cmd = [
        sys.executable,
        "scripts/run_thesis_decision_ladder.py",
        "--label", run_label,
        "--output-root", str(output_root / "static_ladder"),
        "--levels", *cfg["levels"],
        "--garrido-cfis", cfg["garrido_cfis"],
        "--replications", str(cfg["replications"]),
        "--reward-mode", REWARD_MODE,
        "--risk-level", RISK_LEVEL,
        "--observation-version", OBSERVATION_VERSION,
        "--observation-mode", OBSERVATION_MODE,
        "--step-size-hours", str(STEP_SIZE_HOURS),
        "--max-steps", str(cfg["max_steps"]),
    ]
    if STOCHASTIC_PT:
        cmd.append("--stochastic-pt")
    if "l1b_screening_replications" in cfg:
        cmd += ["--l1b-screening-replications", str(cfg["l1b_screening_replications"])]
    if "l1b_top_k" in cfg:
        cmd += ["--l1b-top-k", str(cfg["l1b_top_k"])]
    run_cmd(cmd, cwd=repo_root)
    run_dirs.append(output_root / "static_ladder" / run_label)

elif cfg["kind"] == "ppo":
    for seed in cfg["seeds"]:
        run_label = f"{RUN_PROFILE}_seed_{seed}_{run_stamp}"
        cmd = [
            sys.executable,
            "scripts/run_thesis_decision_ppo_smoke.py",
            "--label", run_label,
            "--output-root", str(output_root / "ppo_adaptive"),
            "--algo", "ppo_mlp",
            "--action-space-mode", cfg["action_space_mode"],
            "--inventory-period-mode", cfg["inventory_period_mode"],
            "--observation-version", OBSERVATION_VERSION,
            "--observation-mode", OBSERVATION_MODE,
            "--reward-mode", REWARD_MODE,
            "--risk-level", RISK_LEVEL,
            "--step-size-hours", str(STEP_SIZE_HOURS),
            "--max-steps", str(cfg["max_steps"]),
            "--train-timesteps", str(cfg["timesteps"]),
            "--eval-episodes", str(cfg["eval_episodes"]),
            "--seed", str(seed),
            "--garrido-cfis", GARRIDO_CFIS,
            "--include-static-grid",
            "--eval-ai-on-garrido-cfis",
            "--learn-initial-decision",
            "--policy-net-arch", "medium",
        ]
        if STOCHASTIC_PT:
            cmd.append("--stochastic-pt")
        run_cmd(cmd, cwd=repo_root)
        run_dirs.append(output_root / "ppo_adaptive" / run_label)
else:
    raise ValueError(f"Unknown profile kind: {cfg['kind']}")

print("run_dirs:")
for path in run_dirs:
    print(" -", path)


In [ ]:
# ===========================
# 4) Summarize outputs
# ===========================

for run_dir in run_dirs:
    print("\n===", run_dir.name, "===")
    summary_path = run_dir / "summary.json"
    policy_path = run_dir / "policy_summary.csv"
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        print("summary keys:", sorted(summary.keys())[:20])
        if "best_policy_by_fill_rate" in summary:
            print("best_policy_by_fill_rate:")
            print(json.dumps(summary["best_policy_by_fill_rate"], indent=2))
        if "best_static_grid_by_fill_rate" in summary:
            print("best_static_grid_by_fill_rate:")
            print(json.dumps(summary["best_static_grid_by_fill_rate"], indent=2))
    if policy_path.exists():
        df = pd.read_csv(policy_path)
        sort_col = "fill_rate_order_level_mean" if "fill_rate_order_level_mean" in df.columns else df.columns[0]
        display(df.sort_values(sort_col, ascending=False).head(20))

In [ ]:
# ==========================================
# 5) Claim paragraph for manuscript notes
# ==========================================

claim_paragraph = (
    "The action space uses the thesis decision variables from Tables 6.16 and 6.20, "
    "including their exact value sets. We relax the thesis protocol along explicitly "
    "declared axes: inventory and capacity decisions may be combined; the replenishment "
    "period may differ per storage node only in the per-node extension; and PPO policies "
    "may revise decisions at each 168-hour epoch, the smallest replenishment period in "
    "Table 6.16, rather than fixing one ex-ante design for the whole horizon. Each "
    "relaxation is evaluated separately, and adaptive policies are compared against the "
    "best static policy available in the same action space."
)
(output_root / "claim_boundary.txt").write_text(claim_paragraph)
print(claim_paragraph)
print("wrote:", output_root / "claim_boundary.txt")
